### Ce notebook permet de faire une vérification de la qualité des données avant toute consolidation, visualisation ou modélisation.

In [2]:
import pandas as pd
from pathlib import Path

###  Chargement de données

In [3]:
# Chemin d'accès aux données brutes

RAW_DATA_DIR = Path("../..") / "data" / "raw"

# Charger les datasets

files = {
    "idmc": "data_idmc_depuis_2000.csv",
    "solutions": "data_solutions_depuis_2000.csv",
    "decisions": "decisions_asile_depuis_2000.csv",
    "demandes": "demandes_asile_depuis_2000.csv",
    "demographie": "demographie_depuis_2000.csv",
    "pays": "countries.csv"
}


dfs = {
    name: pd.read_csv(RAW_DATA_DIR / filename)
    for name, filename in files.items()
}

for name, df in dfs.items():
    print(f"{name:15} : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")

idmc            : 934 lignes × 10 colonnes
solutions       : 21,258 lignes × 13 colonnes
decisions       : 113,929 lignes × 17 colonnes
demandes        : 120,597 lignes × 14 colonnes
demographie     : 116,781 lignes × 24 colonnes
pays            : 232 lignes × 16 colonnes


### Timeliness : couverture temporelle des sources

In [4]:
coverage = []

for name in [
    "idmc",
    "demandes",
    "decisions",
    "solutions",
    "demographie"
]:

    year = pd.to_numeric(
        dfs[name]["year"],
        errors="coerce"
    )

    coverage.append({
        "dataset": name,
        "min": year.min(),
        "max": year.max(),
        "nb_annees": year.nunique(),
        "nb_lignes": len(dfs[name])
    })

coverage = pd.DataFrame(coverage)

display(coverage)

,dataset,min,max,nb_annees,nb_lignes
0,idmc,1990,2025,36,934
1,demandes,2000,2025,26,120597
2,decisions,2000,2025,26,113929
3,solutions,1959,2025,67,21258
4,demographie,2001,2025,25,116781


### Vérification des écarts dans la colonne "year"

In [5]:
idmc = dfs["idmc"].copy()

idmc_year = (
    idmc
    .sort_values(["coo_id", "year"])
    .copy()
)

idmc_year["previous_year"] = (
    idmc_year
    .groupby("coo_id")["year"]
    .shift(1)
)

idmc_year["year_gap"] = (
    idmc_year["year"]
    - idmc_year["previous_year"]
)

# Compter la fréquence des écarts
display(
    idmc_year["year_gap"]
    .value_counts()
    .sort_index()
)

year_gap
1.0    835
2.0      6
3.0      2
4.0      3
5.0      1
6.0      1
7.0      1
9.0      1
Name: count, dtype: int64

La très grande majorité des observations successives sont espacées exactement d’un an, 
mais il existe quelques ruptures temporelles qu’il faut investiguer.


98,2 % des 850 transitions temporelles observées sont annuelles. 15 transitions (1,8 %) présentent une discontinuité supérieure à un an, avec un écart maximal de 9 ans. Ces cas doivent être investigués avant toute imputation ou exclusion.

### Recherche des coo_id et des années avec rupture

In [6]:
gaps = (
    idmc_year
    .loc[idmc_year["year_gap"] > 1,
         ["coo_id", "previous_year", "year", "year_gap"]]
    .sort_values(["year_gap", "coo_id"], ascending=[False, True])
)

display(gaps)

,coo_id,previous_year,year,year_gap
474,262,2008.0,2017,9.0
560,100,2012.0,2019,7.0
757,192,2016.0,2022,6.0
783,73,2018.0,2023,5.0
591,10,2016.0,2020,4.0
930,188,2021.0,2025,4.0
699,262,2017.0,2021,4.0
676,117,2018.0,2021,3.0
853,117,2021.0,2024,3.0
733,100,2020.0,2022,2.0


### L'historique complet des coo_id concernés

In [7]:
coo_with_gaps = (
    idmc_year
    .loc[idmc_year["year_gap"] > 1, "coo_id"]
    .unique()
)

display(
    idmc_year[
        idmc_year["coo_id"].isin(coo_with_gaps)
    ]
    .sort_values(["coo_id", "year"])
)

,year,coo_id,coo_name,coo,coo_iso,coa_id,coa_name,coa,coa_iso,total,previous_year,year_gap
20,2009,10,Armenia,ARM,ARM,10,Armenia,ARM,ARM,8400,NaN,NaN
66,2010,10,Armenia,ARM,ARM,10,Armenia,ARM,ARM,8400,2009.0,1.0
113,2011,10,Armenia,ARM,ARM,10,Armenia,ARM,ARM,8400,2010.0,1.0
162,2012,10,Armenia,ARM,ARM,10,Armenia,ARM,ARM,8400,2011.0,1.0
211,2013,10,Armenia,ARM,ARM,10,Armenia,ARM,ARM,8400,2012.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...
17,2007,262,Unknown,UKN,UNK,262,Unknown,UKN,UNK,26000000,2006.0,1.0
18,2008,262,Unknown,UKN,UNK,262,Unknown,UKN,UNK,26000000,2007.0,1.0
474,2017,262,Unknown,UKN,UNK,262,Unknown,UKN,UNK,2,2008.0,9.0
699,2021,262,Unknown,UKN,UNK,262,Unknown,UKN,UNK,8,2017.0,4.0


Parmi 850 transitions observées entre deux enregistrements successifs d'un même pays d'origine, 835 (98,2 %) sont espacées d'un an et 15 (1,8 %) présentent un intervalle supérieur à un an, avec un maximum de 9 ans. L'analyse des cas concernés montre que ces ruptures peuvent correspondre à des périodes sans observation IDMC pour le pays considéré. Elles sont donc conservées et ne sont pas imputées comme des données manquantes.